>Test cấu trúc

In [13]:
import json

FILE_PATH = "generations.jsonl"

with open(FILE_PATH, "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i >= 5:
            break
        obj = json.loads(line)
        print(obj.keys())
        # print(obj)

dict_keys(['run_id', 'question_idx', 'question', 'hint', 'label', 'model_output', 'output_token_length', 'finish_reason'])
dict_keys(['run_id', 'question_idx', 'question', 'hint', 'label', 'model_output', 'output_token_length', 'finish_reason'])
dict_keys(['run_id', 'question_idx', 'question', 'hint', 'label', 'model_output', 'output_token_length', 'finish_reason'])
dict_keys(['run_id', 'question_idx', 'question', 'hint', 'label', 'model_output', 'output_token_length', 'finish_reason'])
dict_keys(['run_id', 'question_idx', 'question', 'hint', 'label', 'model_output', 'output_token_length', 'finish_reason'])


In [4]:
import json

FILE_PATH = "generations.jsonl"

def split_thinking(text):
    if "</think>" in text:
        think, rest = text.split("</think>", 1)
        think = think.replace("<think>", "").strip()
        return think, rest.strip()
    return "", text

with open(FILE_PATH, "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i >= 5:
            break

        obj = json.loads(line)

        thinking, answer = split_thinking(obj["model_output"])

        print("\n" + "="*80)
        print(f"QUESTION_IDX: {obj['question_idx']}")
        print("-"*80)

        print("THINKING:")
        print(thinking)   # cắt 300 ký tự cho gọn

        print("-"*80)

        print("FINAL:")
        print(answer)


QUESTION_IDX: 00077f9cf9
--------------------------------------------------------------------------------
THINKING:
Okay, so I need to find the positive difference between the measures of two supplementary angles that are in the ratio of 7:2. Hmm, let me start by recalling what supplementary angles are. Supplementary angles are two angles that add up to 180 degrees. So, if they are supplementary, their measures should satisfy the equation: angle1 + angle2 = 180 degrees.

The problem also mentions that the measures of these angles are in the ratio of 7:2. Let me denote the measures of the two angles as 7x and 2x, where x is some positive real number. This ratio of 7:2 tells me that the first angle is 7 parts and the second is 2 parts. 

Since they are supplementary, their sum should be 180 degrees. So, substituting the expressions I have:

7x + 2x = 180

Combining like terms:

9x = 180

Then, solving for x:

x = 180 / 9

Calculating that, 180 divided by 9 is 20. So, x is 20 degrees. 



>Thống kê reason_stop

In [5]:
import json
from collections import Counter

FILE_PATH = "generations.jsonl"

counter = Counter()

with open(FILE_PATH, "r", encoding="utf-8") as f:
    for line in f:
        obj = json.loads(line)
        finish = obj.get("finish_reason", "unknown")
        counter[finish] += 1

print("Finish reason stats:")
for k, v in counter.items():
    print(f"{k}: {v}")

Finish reason stats:
stop: 6380
length: 918


>Số lượng output > 4000

In [10]:
import json

FILE_PATH = "generations.jsonl"

count = 0
total = 0

with open(FILE_PATH, "r", encoding="utf-8") as f:
    for line in f:
        obj = json.loads(line)
        total += 1

        if obj.get("output_token_length", 0) > 2000:
            count += 1

print(f">4000: {count}/{total} ({count/total*100:.2f}%)")

>4000: 2817/7298 (38.60%)


>Chiều dài trung bình khi `stop`, chứng tỏ có những trường hợp đề bài quá dài

In [3]:
import json

FILE_PATH = "generations.jsonl"

length_tokens = []

with open(FILE_PATH, "r", encoding="utf-8") as f:
    for line in f:
        obj = json.loads(line)

        if obj.get("finish_reason") == "length":
            length_tokens.append(obj.get("output_token_length", 0))

# stats
if length_tokens:
    print(f"Total length samples: {len(length_tokens)}")
    print(f"Min: {min(length_tokens)}")
    print(f"Max: {max(length_tokens)}")
    print(f"Avg: {sum(length_tokens)/len(length_tokens):.2f}")
else:
    print("No length samples found")

Total length samples: 918
Min: 7480
Max: 8120
Avg: 8022.14


>Debug, kiểm tra các câu có input dài bất thường

In [ ]:
import json

FILE_PATH = "generations.jsonl"

length_samples = []

with open(FILE_PATH, "r", encoding="utf-8") as f:
    for line in f:
        obj = json.loads(line)

        if obj.get("finish_reason") == "length":
            length_samples.append({
                "question_idx": obj["question_idx"],
                "output_token_length": obj.get("output_token_length", 0),
                "question_hint": obj["full_text"],
            })

# sort tăng dần theo output_token_length
length_samples.sort(key=lambda x: x["output_token_length"])

# lấy top 10 nhỏ nhất
top10 = length_samples[:10]

print("\n===== TOP 10 SMALLEST LENGTH-STOP SAMPLES =====")

for i, sample in enumerate(top10):
    print("\n" + "-"*80)
    print(f"Rank {i+1}")
    print(f"QUESTION_IDX: {sample['question_idx']}")
    print(f"TOKENS      : {sample['output_token_length']}")
    print("QUESTION_HINT:")
    print(sample["question_hint"])  



===== TOP 10 SMALLEST LENGTH-STOP SAMPLES =====

--------------------------------------------------------------------------------
Rank 1
QUESTION_IDX: 6fbb87994d
TOKENS      : 7480
QUESTION:
A frustum of a right circular cone is formed by cutting a small cone off of the top of a larger cone. If a particular frustum has an altitude of $24$ centimeters, the area of its lower base is $225\pi$ sq cm and the area of its upper base is $25\pi$ sq cm, what is the altitude of the small cone that was cut off? [asy]size(200);
import three; defaultpen(linewidth(1)); currentprojection = orthographic(0,-3,0.5); pen dots = linetype("0 3") + linewidth(1);
real h = 2.3, ratio = (91-24)/(171-24);
picture p1, p2; /* p1 is left-hand picture */
triple A = (0,0,0), B = (0,0,h); draw(p1,(-1,0,0)..(0,-1,0)..(1,0,0)); draw(p1,(-1,0,0)..(0,1,0)..(1,0,0),dots); draw(p1,(-1,0,0)--B--(1,0,0));
add(p1);

triple vlift = (0,0,0.5);

path3 toparc1 = shift((0,0,h*(1-ratio)))*scale3(ratio)*((-1,0,0)..(0,1,0)..(1,0,0)),

>Chiều dài trung bình khi `length`, chứng tỏ có trường hợp bị out suy nghĩ

In [4]:
import json

FILE_PATH = "generations.jsonl"

length_tokens = []

with open(FILE_PATH, "r", encoding="utf-8") as f:
    for line in f:
        obj = json.loads(line)

        if obj.get("finish_reason") == "stop":
            length_tokens.append(obj.get("output_token_length", 0))

# stats
if length_tokens:
    print(f"Total length samples: {len(length_tokens)}")
    print(f"Min: {min(length_tokens)}")
    print(f"Max: {max(length_tokens)}")
    print(f"Avg: {sum(length_tokens)/len(length_tokens):.2f}")
else:
    print("No length samples found")

Total length samples: 6380
Min: 9
Max: 8057
Avg: 1765.85
